# NB5 -- Defence 1: Adversarially-Trained XGBoost (Multiclass, v4)

Rebuilt against the v4 corrected pipeline from the original leakage-era
`NB5_Defence1_Drive_results.ipynb` architectural reference. Only the data
pipeline, target-model source, and the items below are corrected; the core
adversarial-training design (1:1 clean:adversarial augmentation, same tuned
hyperparameters, true-label strategy) is carried over unchanged.

**Corrections applied vs. the original reference notebook:**
- Data: loads `phase1_data_v4.pkl` (temporal+embargo split, 43 features), local
  Windows paths, GPU-detection pattern matching NB2/NB3/NB4.
- Target model: retrains the Optuna-tuned XGBoost from the actual v4
  `best_params` (multiclass uses tuned, matching NB3/NB4's asymmetry) --
  no fitted tuned-booster artifact exists on disk.
- **Test-set PGD reused, not regenerated**: loads NB3's saved
  `X_adv_pgd_eps01_v4.npy` directly instead of regenerating with a fresh
  surrogate, so Defence 1's PGD numbers are evaluated against the *exact same*
  adversarial examples Attack 1 reported on, not a near-duplicate.
- **Training-set PGD still regenerated in-notebook** (unavoidable -- no
  training-set PGD artifact exists from Attack 1, which only saved test-set
  examples), using the same EPS=0.1, ALPHA=EPS/4, N_ITER=10 recipe.
- **PPO threat model (per project decision)**: the augmented training set uses
  only the **budget-matched** PPO agent (`ppo_evasion_agent_budgeted_v4.zip`,
  eps_total_budget=0.1 -- matched to PGD, the corrected primary finding from
  NB4 Sec. 7b/7). The **unconstrained** agent is used only at evaluation time,
  as a stress test of whether the defence generalizes beyond the budget it was
  trained against, or overfits narrowly to it.
- **Full-set PPO re-evaluation for both agents against both models**: the
  original notebook's master-comparison table silently reused
  `X_adv_ppo_full.npy` (generated by attacking the *standard* model only) as
  if it were a fair re-attack on the *defended* model -- conflating
  transferability with a genuine re-attack. This version regenerates a proper
  full-attack-class-test-set PPO rollout separately for
  (unconstrained, budget-matched) x (standard, defended) = 4 combinations, so
  every cell in the master table is a genuine model-specific attack, not a
  reused/transferred one.
- Dropped the reference's ad hoc n=500/class PPO subsample for the recovery
  table in favour of the full test set, consistent with NB4's precedent.

**Unchanged from the reference architecture:**
- Fresh in-notebook SurrogateMLP (same architecture as Attack 1) for PGD
  generation.
- Loads NB4's already-trained PPO agent(s) rather than retraining (this fix
  was already present in the reference notebook, with its own note explaining
  why: retraining in-place used to silently overwrite NB4's saved agent,
  making Defence 1 evaluate against a different attacker than Attack 2
  reported on).
- 1:1 clean:adversarial augmentation ratio; true labels on all adversarial
  examples; same tuned hyperparameters as the standard IDS.

**Prerequisite**: run the NB4 addendum first (saves
`ppo_evasion_agent_budgeted_v4.zip`) if not already done.


## Setup

In [ ]:
!pip install -q sb3-contrib gymnasium


In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from sklearn.metrics import f1_score
import gymnasium as gym
from gymnasium import spaces
from sb3_contrib import RecurrentPPO
warnings.filterwarnings('ignore')

# Drive-ready: PHYSAR-NEW lives on Google Drive (survives Colab runtime restarts)
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/PHYSAR-NEW'
os.chdir(PROJECT_DIR)
if not os.path.exists('phase1_data_v4.pkl'):
    print(f'WARNING: phase1_data_v4.pkl not found in {PROJECT_DIR} -- fix PROJECT_DIR before continuing.')
    print(sorted(os.listdir(PROJECT_DIR))[:20])
else:
    print('Found phase1_data_v4.pkl -- PROJECT_DIR looks correct.')

RANDOM_STATE = 42
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
torch.cuda.manual_seed_all(RANDOM_STATE)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Working directory: {os.getcwd()}')
print(f'Device: {DEVICE}')
print(f'Seeds set to {RANDOM_STATE}')

# GPU detection for XGBoost -- same pattern as NB2/NB3/NB4
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _test_model = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2)
    _test_model.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _fell_back_to_cpu = any('No visible GPU' in str(w.message) for w in _caught)

if _fell_back_to_cpu:
    GPU_PARAMS = {}
    print('GPU not available for XGBoost (silently fell back to CPU) -- using CPU.')
else:
    GPU_PARAMS = {'tree_method': 'hist', 'device': 'cuda'}
    print('GPU detected -- XGBoost will use CUDA (tree_method=hist, device=cuda).')


## 1. Load Phase 1 data and the tuned XGBoost hyperparameters

In [ ]:
PIPELINE_VERSION = 'v4'
EXPECTED_TEST_FRAC = 0.15
EXPECTED_EMBARGO_SECONDS = 1.0

data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')

version_mismatch = (
    data.get('PIPELINE_VERSION') != PIPELINE_VERSION
    or data.get('TEST_FRAC') != EXPECTED_TEST_FRAC
    or data.get('EMBARGO_SECONDS') != EXPECTED_EMBARGO_SECONDS
)
if version_mismatch:
    print("WARNING: loaded file's version metadata doesn't match this notebook's "
          "expectations -- proceeding anyway, but double check before trusting results.")
else:
    print(f'Version check passed: {PIPELINE_VERSION}, TEST_FRAC={EXPECTED_TEST_FRAC}, '
          f'EMBARGO_SECONDS={EXPECTED_EMBARGO_SECONDS}')

X_train_scaled = data['X_train_scaled']
X_test_scaled  = data['X_test_scaled']
y_train = pd.Series(data['y_train'])
y_test  = pd.Series(data['y_test'])
FEATURE_COLS = data['FEATURE_COLS']

print(f'Train: {X_train_scaled.shape}, Test: {X_test_scaled.shape}')
print(f'Features: {len(FEATURE_COLS)}')

# CHANGED: this notebook now targets the UNTUNED baseline XGBoost (Table 6.1's
# headline model, F1=0.7225) rather than the Optuna-tuned configuration (F1=0.7110)
# the original NB5 used -- companion to the Attack 1/2 baseline checks.
baseline_results = joblib.load(f'baseline_results_{PIPELINE_VERSION}.pkl')
best_params = dict(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
)
print(f"Using untuned baseline hyperparameters (recorded test F1: "
      f"{baseline_results['xgb_test_f1']:.4f})")


## 2. Standard (undefended) target model: retrain the tuned XGBoost

In [ ]:
best_params.update({
    'objective': 'multi:softprob',
    'num_class': 6,
    'eval_metric': 'mlogloss',
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    **GPU_PARAMS,
})

xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_train_scaled, y_train, verbose=False)

clean_test_f1_standard = f1_score(y_test, xgb_model.predict(X_test_scaled), average='weighted')
print(f'Standard XGBoost -- Test F1: {clean_test_f1_standard:.4f}')

recorded_test_f1 = baseline_results['xgb_test_f1']
f1_diff = abs(clean_test_f1_standard - recorded_test_f1)
if f1_diff > 0.005:
    print(f"WARNING: retrained test F1 differs from recorded value by {f1_diff:.4f} -- "
          f"double-check GPU_PARAMS / library versions.")
else:
    print(f'Sanity check passed: retrained test F1 matches recorded test F1 (diff={f1_diff:.4f}).')

CYBER_FEATURE_NAMES = [
    'cyber_pkt_count', 'cyber_pktlen_mean', 'cyber_pktlen_std',
    'cyber_n_unique_srcport', 'cyber_n_unique_dstport',
    'cyber_modbus_frac', 'cyber_tls_frac', 'cyber_tcp_frac',
    'cyber_iat_mean', 'cyber_iat_std',
]
CYBER_IDX = np.array([FEATURE_COLS.index(c) for c in CYBER_FEATURE_NAMES])
N_FEATURES = len(FEATURE_COLS)
N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
print(f'Total features: {N_FEATURES}, attacker-controllable (cyber) features: {N_CYBER}')


## 3. Attack 1 (PGD) artifacts

Test-set PGD examples are **loaded directly from Attack 1's saved artifact**
(not regenerated) so Defence 1's PGD evaluation is against the exact same
examples Attack 1 reported on. Training-set PGD examples still need
regenerating here (no such artifact exists from Attack 1).

In [ ]:
class SurrogateMLP(nn.Module):
    def __init__(self, n_features, n_classes=6, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden // 2), nn.ReLU(),
            nn.Linear(hidden // 2, n_classes),
        )
    def forward(self, x):
        return self.net(x)

n_feat = X_train_scaled.shape[1]
surrogate = SurrogateMLP(n_feat).to(DEVICE)
sur_opt = torch.optim.Adam(surrogate.parameters(), lr=1e-3)
sur_criterion = nn.CrossEntropyLoss()

X_t = torch.from_numpy(X_train_scaled).float()
y_t = torch.from_numpy(y_train.values).long()
sur_loader = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)

for epoch in range(20):
    surrogate.train()
    for X_batch, y_batch in sur_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        sur_opt.zero_grad()
        loss = sur_criterion(surrogate(X_batch), y_batch)
        loss.backward()
        sur_opt.step()
print('Surrogate trained.')

EPS, ALPHA, N_ITER = 0.1, 0.1 / 4, 10

def pgd_generate(model, X, y_np, epsilon, alpha, n_iter):
    model.eval()
    y_t_ = torch.from_numpy(y_np).long().to(DEVICE)
    X_orig = torch.from_numpy(X).float().to(DEVICE)
    X_adv = X_orig.clone().detach()
    for _ in range(n_iter):
        X_adv.requires_grad_(True)
        loss = F.cross_entropy(model(X_adv), y_t_)
        grad = torch.autograd.grad(loss, X_adv)[0]
        X_adv = X_adv.detach() + alpha * grad.sign()
        perturbation = torch.clamp(X_adv - X_orig, -epsilon, epsilon)
        X_adv = (X_orig + perturbation).detach()
    return X_adv.cpu().numpy()

BATCH = 4096
X_adv_pgd_train = np.zeros_like(X_train_scaled)
y_train_np = y_train.values
for start in range(0, len(X_train_scaled), BATCH):
    end = min(start + BATCH, len(X_train_scaled))
    X_adv_pgd_train[start:end] = pgd_generate(
        surrogate, X_train_scaled[start:end], y_train_np[start:end], EPS, ALPHA, N_ITER)
print(f'PGD adversarial training examples generated: {X_adv_pgd_train.shape}')

# CHANGED: load Attack 1's BASELINE-model PGD test examples, not the tuned-model ones.
X_adv_pgd_test = np.load(f'X_adv_pgd_eps01_baseline_{PIPELINE_VERSION}.npy')
y_test_np = y_test.values
print(f'PGD adversarial test examples loaded from Attack 1 (baseline model): {X_adv_pgd_test.shape}')


## 4. Attack 2 (PPO) artifacts

Loads **both** PPO agents from NB4 (does not retrain either -- retraining here
would silently diverge from what Attack 2 reported on). The **budget-matched**
agent supplies the augmentation training examples; the **unconstrained**
agent is used only later, as an evaluation-time stress test.

In [ ]:
EPS_BOUND = 0.3
MAX_STEPS = 10

class IDSEvasionEnv(gym.Env):
    """Black-box evasion environment -- identical to NB4's, including the
    optional cumulative eps_total_budget clamp used for the budget-matched
    agent."""
    metadata = {'render_modes': []}

    def __init__(self, X_pool, y_pool, model, cyber_idx, n_features, n_classes=6,
                 eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None,
                 eps_total_budget=None):
        super().__init__()
        self.X_pool = X_pool
        self.y_pool = y_pool
        self.model = model
        self.cyber_idx = np.array(cyber_idx)
        self.n_features = n_features
        self.n_cyber = len(cyber_idx)
        self.n_classes = n_classes
        self.eps_bound = eps_bound
        self.max_steps = max_steps
        self.eps_total_budget = eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(self.n_cyber,), dtype=np.float32)
        obs_dim = self.n_features + self.n_cyber + self.n_classes + 1
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(obs_dim,), dtype=np.float32)

    def _get_obs(self):
        last_pred_onehot = np.zeros(self.n_classes, dtype=np.float32)
        last_pred_onehot[self.last_pred] = 1.0
        step_norm = np.array([self.step_count / self.max_steps], dtype=np.float32)
        return np.concatenate([self.x_current.astype(np.float32),
                                self.last_perturbation.astype(np.float32),
                                last_pred_onehot, step_norm])

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy()
        self.x_current = self.x_orig.copy()
        self.y_true = self.y_pool[idx]
        self.step_count = 0
        self.last_perturbation = np.zeros(self.n_cyber, dtype=np.float32)
        self.last_pred = self.model.predict(self.x_current.reshape(1, -1))[0]
        return self._get_obs(), {}

    def step(self, action):
        action = np.clip(action, -self.eps_bound, self.eps_bound)
        if self.eps_total_budget is not None:
            proposed = (self.x_current[self.cyber_idx] + action) - self.x_orig[self.cyber_idx]
            proposed = np.clip(proposed, -self.eps_total_budget, self.eps_total_budget)
            self.x_current[self.cyber_idx] = self.x_orig[self.cyber_idx] + proposed
        else:
            self.x_current[self.cyber_idx] = self.x_current[self.cyber_idx] + action
        self.last_perturbation = action.astype(np.float32)
        self.step_count += 1
        pred = self.model.predict(self.x_current.reshape(1, -1))[0]
        self.last_pred = pred
        terminated = truncated = False
        if pred == 0:
            reward = 10.0 - 0.01 * np.sum(action ** 2)
            terminated = True
        elif self.step_count >= self.max_steps:
            reward = -1.0 - 0.01 * np.sum(action ** 2)
            truncated = True
        else:
            reward = -0.01 * np.sum(action ** 2)
        return self._get_obs(), reward, terminated, truncated, {'evaded': pred == 0, 'x_current': self.x_current.copy()}

attack_mask_train = (y_train.values != 0)
X_attack_train = X_train_scaled[attack_mask_train]
y_attack_train = y_train.values[attack_mask_train]

attack_mask_test = (y_test_np != 0)
X_attack_test = X_test_scaled[attack_mask_test]
y_attack_test = y_test_np[attack_mask_test]

env_for_loading = IDSEvasionEnv(X_pool=X_attack_train, y_pool=y_attack_train, model=xgb_model,
                                 cyber_idx=CYBER_IDX, n_features=N_FEATURES, seed=RANDOM_STATE)

# CHANGED: load Attack 2's BASELINE-model PPO agents, not the tuned-model ones.
ppo_model_budgeted = RecurrentPPO.load('ppo_evasion_agent_budgeted_baseline_v4', env=env_for_loading)
print('Loaded budget-matched PPO agent (NB4 baseline) -- not retrained.')

ppo_model_unconstrained = RecurrentPPO.load('ppo_evasion_agent_unconstrained_baseline_v4', env=env_for_loading)
print('Loaded unconstrained PPO agent (NB4 baseline) -- not retrained.')


## 5. Collect PPO adversarial training examples (budget-matched agent, training pool)

In [ ]:
N_PER_CLASS_PPO = 1000  # reduce if too slow; increase for a larger adversarial pool

def collect_ppo_examples(ppo_agent, X_pool, y_pool, model, n_per_class, eps_total_budget, seed=0):
    X_adv, y_adv = [], []
    for cls_idx in range(1, 6):
        cls_mask = np.where(y_pool == cls_idx)[0]
        chosen = np.random.RandomState(seed).choice(cls_mask, size=min(n_per_class, len(cls_mask)), replace=False)
        X_cls, y_cls = X_pool[chosen], y_pool[chosen]
        for i in range(len(X_cls)):
            env_tmp = IDSEvasionEnv(X_pool=X_cls[i:i+1], y_pool=y_cls[i:i+1], model=model,
                                     cyber_idx=CYBER_IDX, n_features=N_FEATURES, seed=0,
                                     eps_total_budget=eps_total_budget)
            obs, _ = env_tmp.reset()
            lstm_states, episode_starts = None, np.ones((1,), dtype=bool)
            x_final = X_cls[i].copy()
            for _ in range(MAX_STEPS):
                action, lstm_states = ppo_agent.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
                episode_starts = np.zeros((1,), dtype=bool)
                obs, _, terminated, truncated, info = env_tmp.step(action)
                x_final = info['x_current']
                if terminated or truncated:
                    break
            X_adv.append(x_final)
            y_adv.append(cls_idx)
        print(f'  Class {cls_idx} ({class_names[cls_idx]}): {len(chosen)} PPO samples collected')
    return np.array(X_adv, dtype=np.float32), np.array(y_adv)

print('Collecting PPO training examples from the BUDGET-MATCHED agent (training pool)...')
X_ppo_adv, y_ppo_adv = collect_ppo_examples(
    ppo_model_budgeted, X_attack_train, y_attack_train, xgb_model,
    N_PER_CLASS_PPO, eps_total_budget=0.1, seed=RANDOM_STATE)
print(f'PPO adversarial training examples: {X_ppo_adv.shape}, class distribution: {np.bincount(y_ppo_adv)}')


## 6. Build augmented training dataset (1:1 clean:adversarial)

In [ ]:
X_adv_all = np.vstack([X_adv_pgd_train, X_ppo_adv])
y_adv_all = np.concatenate([y_train_np, y_ppo_adv])

print(f'Clean training examples:    {X_train_scaled.shape[0]}')
print(f'PGD adversarial examples:   {X_adv_pgd_train.shape[0]}')
print(f'PPO adversarial examples:   {X_ppo_adv.shape[0]} (budget-matched)')
print(f'Total adversarial pool:     {X_adv_all.shape[0]}')

n_clean = len(X_train_scaled)
n_adv = len(X_adv_all)
if n_adv > n_clean:
    idx = np.random.RandomState(RANDOM_STATE).choice(n_adv, size=n_clean, replace=False)
    X_adv_sampled, y_adv_sampled = X_adv_all[idx], y_adv_all[idx]
else:
    X_adv_sampled, y_adv_sampled = X_adv_all, y_adv_all

X_aug = np.vstack([X_train_scaled, X_adv_sampled])
y_aug = np.concatenate([y_train_np, y_adv_sampled])
perm = np.random.RandomState(RANDOM_STATE).permutation(len(X_aug))
X_aug, y_aug = X_aug[perm], y_aug[perm]

print(f'Augmented training set: {X_aug.shape}')
print(f'Class distribution: {np.bincount(y_aug)}')
print(f'Clean:Adversarial ratio = 1:{X_adv_sampled.shape[0]/n_clean:.2f}')


## 7. Train the adversarially-defended XGBoost

In [ ]:
xgb_defended = xgb.XGBClassifier(**best_params)
xgb_defended.fit(X_aug, y_aug, verbose=False)

clean_train_f1 = f1_score(y_train, xgb_defended.predict(X_train_scaled), average='weighted')
clean_test_f1_defended = f1_score(y_test, xgb_defended.predict(X_test_scaled), average='weighted')

print('Defended XGBoost -- clean data evaluation:')
print(f'  Train F1: {clean_train_f1:.4f}')
print(f'  Test F1:  {clean_test_f1_defended:.4f}')
print(f'  Gap:      {clean_train_f1 - clean_test_f1_defended:.4f}')
print(f'\nStandard XGBoost clean Test F1 (v4, untuned baseline): {clean_test_f1_standard:.4f}')
print(f'Clean accuracy cost of adversarial training: {clean_test_f1_standard - clean_test_f1_defended:+.4f}')


## 8. Evaluate under Attack 1 (PGD) -- Standard vs. Defended

In [ ]:
pred_pgd_standard = xgb_model.predict(X_adv_pgd_test)
pred_pgd_defended = xgb_defended.predict(X_adv_pgd_test)

print('PGD (epsilon=0.1) -- Standard vs. Defended XGBoost:')
print(f"{'Class':<14} {'Standard Evasion':>18} {'Defended Evasion':>18} {'Recovery':>10}")
for cls_idx, cls_name in enumerate(class_names[1:], start=1):
    mask = (y_test_np == cls_idx)
    std_evasion = (pred_pgd_standard[mask] == 0).mean()
    def_evasion = (pred_pgd_defended[mask] == 0).mean()
    print(f"{cls_name:<14} {std_evasion:>18.4f} {def_evasion:>18.4f} {std_evasion-def_evasion:>+10.4f}")


## 9. Evaluate under Attack 2 (PPO) -- full test-set rollout, both agents, both models

Regenerates a genuine full-attack-class-test-set rollout for all four
(agent x model) combinations, rather than reusing/transferring a rollout
generated against a different model. This is the compute-heavy cell in this
notebook (4 x ~81,718-sample sweeps, up to 10 model.predict calls each).

In [ ]:
def full_set_ppo_rollout(ppo_agent, model, eps_total_budget):
    evasion = {cls_idx: {'total': 0, 'evaded': 0} for cls_idx in range(1, 6)}
    X_adv_final = np.zeros_like(X_attack_test)
    eval_env = IDSEvasionEnv(X_pool=X_attack_test, y_pool=y_attack_test, model=model,
                              cyber_idx=CYBER_IDX, n_features=N_FEATURES, seed=RANDOM_STATE + 1,
                              eps_total_budget=eps_total_budget)
    for i in range(len(X_attack_test)):
        eval_env.X_pool = X_attack_test[i:i+1]
        eval_env.y_pool = y_attack_test[i:i+1]
        eval_env.rng = np.random.RandomState(0)
        obs, _ = eval_env.reset()
        lstm_states, episode_starts = None, np.ones((1,), dtype=bool)
        true_class = int(y_attack_test[i])
        evasion[true_class]['total'] += 1
        final_obs = obs
        for step in range(MAX_STEPS):
            action, lstm_states = ppo_agent.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
            episode_starts = np.zeros((1,), dtype=bool)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            final_obs = obs
            if info.get('evaded', False):
                evasion[true_class]['evaded'] += 1
                break
            if terminated or truncated:
                break
        X_adv_final[i] = np.asarray(final_obs).reshape(-1)[:N_FEATURES]
    return evasion, X_adv_final

print('Rolling out: unconstrained agent vs standard model...')
evasion_unc_std, X_adv_ppo_unc_std = full_set_ppo_rollout(ppo_model_unconstrained, xgb_model, eps_total_budget=None)
print('Rolling out: unconstrained agent vs defended model...')
evasion_unc_def, X_adv_ppo_unc_def = full_set_ppo_rollout(ppo_model_unconstrained, xgb_defended, eps_total_budget=None)
print('Rolling out: budget-matched agent vs standard model...')
evasion_bud_std, X_adv_ppo_bud_std = full_set_ppo_rollout(ppo_model_budgeted, xgb_model, eps_total_budget=0.1)
print('Rolling out: budget-matched agent vs defended model...')
evasion_bud_def, X_adv_ppo_bud_def = full_set_ppo_rollout(ppo_model_budgeted, xgb_defended, eps_total_budget=0.1)

def rate_table(name, evasion_std, evasion_def):
    print(f"\n{name} -- Standard vs. Defended XGBoost (full test set):")
    print(f"{'Class':<14} {'Standard Evasion':>18} {'Defended Evasion':>18} {'Recovery':>10}")
    for cls_idx in range(1, 6):
        std_rate = evasion_std[cls_idx]['evaded'] / evasion_std[cls_idx]['total']
        def_rate = evasion_def[cls_idx]['evaded'] / evasion_def[cls_idx]['total']
        print(f"{class_names[cls_idx]:<14} {std_rate:>18.4f} {def_rate:>18.4f} {std_rate-def_rate:>+10.4f}")

rate_table('PPO unconstrained', evasion_unc_std, evasion_unc_def)
rate_table('PPO budget-matched (eps=0.1)', evasion_bud_std, evasion_bud_def)


## 10. Master comparison: Standard vs. Robust IDS

Every cell in this table is now a genuine, model-specific attack -- no reused
or transferred adversarial examples.

In [ ]:
def full_f1(X_adv_final_by_class, model):
    X_full = X_test_scaled.copy()
    X_full[attack_mask_test] = X_adv_final_by_class
    return f1_score(y_test, model.predict(X_full), average='weighted')

print('=' * 80)
print('MASTER COMPARISON: Standard IDS vs. Robust (Adversarially-Trained) IDS')
print('=' * 80)
print(f"{'Metric':<38} {'Standard IDS':>15} {'Robust IDS':>15} {'Change':>10}")
print('-' * 80)
print(f"{'Clean Test F1':<38} {clean_test_f1_standard:>15.4f} {clean_test_f1_defended:>15.4f} {clean_test_f1_defended-clean_test_f1_standard:>+10.4f}")

pgd_f1_std = f1_score(y_test, pred_pgd_standard, average='weighted')
pgd_f1_def = f1_score(y_test, pred_pgd_defended, average='weighted')
print(f"{'PGD Test F1 (eps=0.1)':<38} {pgd_f1_std:>15.4f} {pgd_f1_def:>15.4f} {pgd_f1_def-pgd_f1_std:>+10.4f}")

pgd_ev_std = (pred_pgd_standard[attack_mask_test] == 0).mean()
pgd_ev_def = (pred_pgd_defended[attack_mask_test] == 0).mean()
print(f"{'PGD Overall Evasion Rate':<38} {pgd_ev_std:>15.4f} {pgd_ev_def:>15.4f} {pgd_ev_def-pgd_ev_std:>+10.4f}")

ppo_unc_f1_std = full_f1(X_adv_ppo_unc_std, xgb_model)
ppo_unc_f1_def = full_f1(X_adv_ppo_unc_def, xgb_defended)
print(f"{'PPO (unconstrained) Test F1':<38} {ppo_unc_f1_std:>15.4f} {ppo_unc_f1_def:>15.4f} {ppo_unc_f1_def-ppo_unc_f1_std:>+10.4f}")

ppo_unc_ev_std = sum(v['evaded'] for v in evasion_unc_std.values()) / sum(v['total'] for v in evasion_unc_std.values())
ppo_unc_ev_def = sum(v['evaded'] for v in evasion_unc_def.values()) / sum(v['total'] for v in evasion_unc_def.values())
print(f"{'PPO (unconstrained) Evasion Rate':<38} {ppo_unc_ev_std:>15.4f} {ppo_unc_ev_def:>15.4f} {ppo_unc_ev_def-ppo_unc_ev_std:>+10.4f}")

ppo_bud_f1_std = full_f1(X_adv_ppo_bud_std, xgb_model)
ppo_bud_f1_def = full_f1(X_adv_ppo_bud_def, xgb_defended)
print(f"{'PPO (budget-matched) Test F1':<38} {ppo_bud_f1_std:>15.4f} {ppo_bud_f1_def:>15.4f} {ppo_bud_f1_def-ppo_bud_f1_std:>+10.4f}")

ppo_bud_ev_std = sum(v['evaded'] for v in evasion_bud_std.values()) / sum(v['total'] for v in evasion_bud_std.values())
ppo_bud_ev_def = sum(v['evaded'] for v in evasion_bud_def.values()) / sum(v['total'] for v in evasion_bud_def.values())
print(f"{'PPO (budget-matched) Evasion Rate':<38} {ppo_bud_ev_std:>15.4f} {ppo_bud_ev_def:>15.4f} {ppo_bud_ev_def-ppo_bud_ev_std:>+10.4f}")
print('=' * 80)


## Save Defence 1 results to disk

In [ ]:
xgb_defended.save_model(f'xgb_defended_model_baseline_{PIPELINE_VERSION}.json')

defence1_results = {
    'PIPELINE_VERSION': PIPELINE_VERSION,
    'clean_test_f1_standard': clean_test_f1_standard,
    'clean_test_f1_defended': clean_test_f1_defended,
    'clean_accuracy_cost': clean_test_f1_standard - clean_test_f1_defended,
    'pgd_evasion_standard': pgd_ev_std, 'pgd_evasion_defended': pgd_ev_def,
    'pgd_f1_standard': pgd_f1_std, 'pgd_f1_defended': pgd_f1_def,
    'ppo_unconstrained_evasion_standard': ppo_unc_ev_std, 'ppo_unconstrained_evasion_defended': ppo_unc_ev_def,
    'ppo_unconstrained_f1_standard': ppo_unc_f1_std, 'ppo_unconstrained_f1_defended': ppo_unc_f1_def,
    'ppo_budgeted_evasion_standard': ppo_bud_ev_std, 'ppo_budgeted_evasion_defended': ppo_bud_ev_def,
    'ppo_budgeted_f1_standard': ppo_bud_f1_std, 'ppo_budgeted_f1_defended': ppo_bud_f1_def,
    'per_class_pgd': {class_names[c]: {
        'standard': (pred_pgd_standard[y_test_np == c] == 0).mean(),
        'defended': (pred_pgd_defended[y_test_np == c] == 0).mean(),
    } for c in range(1, 6)},
    'per_class_ppo_unconstrained': {class_names[c]: {
        'standard': evasion_unc_std[c]['evaded'] / evasion_unc_std[c]['total'],
        'defended': evasion_unc_def[c]['evaded'] / evasion_unc_def[c]['total'],
    } for c in range(1, 6)},
    'per_class_ppo_budgeted': {class_names[c]: {
        'standard': evasion_bud_std[c]['evaded'] / evasion_bud_std[c]['total'],
        'defended': evasion_bud_def[c]['evaded'] / evasion_bud_def[c]['total'],
    } for c in range(1, 6)},
}
joblib.dump(defence1_results, f'defence1_results_baseline_{PIPELINE_VERSION}.pkl')

print(f'NB5-baseline ({PIPELINE_VERSION}, multiclass, UNTUNED XGBoost) artifacts saved:')
for f in [f'xgb_defended_model_baseline_{PIPELINE_VERSION}.json', f'defence1_results_baseline_{PIPELINE_VERSION}.pkl']:
    if os.path.exists(f):
        print(f'  {f}  ({os.path.getsize(f)/1e6:.1f} MB)')


## Notes

- The **defended model** (`xgb_defended_model_v4.json`) carries forward into:
  Attack 3 (adaptive white-box, surrogate retrained against this boundary),
  Defence 2 (autoencoder gate, tested on top of both standard and defended
  models), and ESI/PCR (SHAP attributions on this model).
- Clean accuracy cost is reported explicitly; if it exceeds ~0.02 F1, consider
  a different augmentation ratio (e.g. 2:1 clean:adversarial) before treating
  this as the locked Robust IDS.
- The comparison between "PPO (budget-matched)" and "PPO (unconstrained)"
  recovery rates is itself a finding: if the defended model's recovery holds
  up against the unconstrained ceiling despite only ever training on
  budget-matched examples, that's evidence of genuine generalization rather
  than narrow overfitting to eps=0.1.


## 11. Seed-robustness check

The single-run Defence 1 result above showed an unusual pattern: budget-matched
adversarial training gave essentially no protection against the unconstrained
PPO ceiling attack, and for **backdoor specifically the evasion rate got worse**
(0.8511 -> 0.9445). Before treating that as a real finding for Chapter 6, it
needs the same seed-robustness check the rest of this pipeline has already
applied to every other load-bearing number (XGBoost tuning, PGD vs. FGSM,
PPO evasion rates).

Per seed, this re-runs the full Defence 1 pipeline: fresh surrogate -> fresh
PGD training examples -> fresh PPO training-example subsample (same
budget-matched agent, not retrained) -> fresh augmented training set -> fresh
defended XGBoost. Clean F1 and PGD evasion are cheap to compute on the full
test set every seed. The PPO evasion re-check uses a **fixed stratified
subsample** (same subsample across all 5 seeds, for a fair comparison),
matching the compute-vs-completeness tradeoff NB4 already established for its
own seed-robustness section, rather than repeating the ~45-minute full-set
rollout five times per agent.

In [ ]:
SEED_LIST = [42, 7, 123, 2024, 99]
N_EVAL_PER_CLASS_SUBSAMPLE = 300  # matches NB4's own seed-robustness subsample size

# Fixed stratified subsample for PPO re-evaluation -- same subsample used for every
# seed's defended model, so differences reflect the defended model changing, not the
# benchmark set changing.
_subsample_rng = np.random.RandomState(999)
eval_idx = []
for cls_idx in range(1, 6):
    cls_mask = np.where(y_attack_test == cls_idx)[0]
    n_take = min(N_EVAL_PER_CLASS_SUBSAMPLE, len(cls_mask))
    eval_idx.append(_subsample_rng.choice(cls_mask, size=n_take, replace=False))
eval_idx = np.concatenate(eval_idx)
X_eval_sub = X_attack_test[eval_idx]
y_eval_sub = y_attack_test[eval_idx]
print(f'Fixed PPO evaluation subsample: {len(eval_idx)} samples, '
      f'class distribution: {np.bincount(y_eval_sub)[1:]}')

def subsample_ppo_evasion(ppo_agent, model, eps_total_budget, X_sub, y_sub):
    evasion = {cls_idx: {'total': 0, 'evaded': 0} for cls_idx in range(1, 6)}
    eval_env = IDSEvasionEnv(X_pool=X_sub, y_pool=y_sub, model=model,
                              cyber_idx=CYBER_IDX, n_features=N_FEATURES,
                              seed=RANDOM_STATE + 1, eps_total_budget=eps_total_budget)
    for i in range(len(X_sub)):
        eval_env.X_pool = X_sub[i:i+1]
        eval_env.y_pool = y_sub[i:i+1]
        eval_env.rng = np.random.RandomState(0)
        obs, _ = eval_env.reset()
        lstm_states, episode_starts = None, np.ones((1,), dtype=bool)
        true_class = int(y_sub[i])
        evasion[true_class]['total'] += 1
        for step in range(MAX_STEPS):
            action, lstm_states = ppo_agent.predict(obs, state=lstm_states, episode_start=episode_starts, deterministic=True)
            episode_starts = np.zeros((1,), dtype=bool)
            obs, reward, terminated, truncated, info = eval_env.step(action)
            if info.get('evaded', False):
                evasion[true_class]['evaded'] += 1
                break
            if terminated or truncated:
                break
    total_evaded = sum(v['evaded'] for v in evasion.values())
    total_n = sum(v['total'] for v in evasion.values())
    per_class = {class_names[c]: (evasion[c]['evaded'] / evasion[c]['total'] if evasion[c]['total'] else 0.0)
                 for c in range(1, 6)}
    return total_evaded / total_n, per_class

seed_results = []
for run_idx, seed in enumerate(SEED_LIST):
    print(f'\n{"="*60}\nSeed {seed} ({run_idx+1}/{len(SEED_LIST)})\n{"="*60}')
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

    surrogate_s = SurrogateMLP(n_feat).to(DEVICE)
    opt_s = torch.optim.Adam(surrogate_s.parameters(), lr=1e-3)
    loader_s = DataLoader(TensorDataset(X_t, y_t), batch_size=256, shuffle=True)
    for epoch in range(20):
        surrogate_s.train()
        for X_batch, y_batch in loader_s:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            opt_s.zero_grad()
            loss = sur_criterion(surrogate_s(X_batch), y_batch)
            loss.backward()
            opt_s.step()

    X_adv_pgd_train_s = np.zeros_like(X_train_scaled)
    for start in range(0, len(X_train_scaled), BATCH):
        end = min(start + BATCH, len(X_train_scaled))
        X_adv_pgd_train_s[start:end] = pgd_generate(
            surrogate_s, X_train_scaled[start:end], y_train_np[start:end], EPS, ALPHA, N_ITER)

    X_ppo_adv_s, y_ppo_adv_s = collect_ppo_examples(
        ppo_model_budgeted, X_attack_train, y_attack_train, xgb_model,
        N_PER_CLASS_PPO, eps_total_budget=0.1, seed=seed)

    X_adv_all_s = np.vstack([X_adv_pgd_train_s, X_ppo_adv_s])
    y_adv_all_s = np.concatenate([y_train_np, y_ppo_adv_s])
    n_adv_s = len(X_adv_all_s)
    if n_adv_s > n_clean:
        idx_s = np.random.RandomState(seed).choice(n_adv_s, size=n_clean, replace=False)
        X_adv_sampled_s, y_adv_sampled_s = X_adv_all_s[idx_s], y_adv_all_s[idx_s]
    else:
        X_adv_sampled_s, y_adv_sampled_s = X_adv_all_s, y_adv_all_s
    X_aug_s = np.vstack([X_train_scaled, X_adv_sampled_s])
    y_aug_s = np.concatenate([y_train_np, y_adv_sampled_s])
    perm_s = np.random.RandomState(seed).permutation(len(X_aug_s))
    X_aug_s, y_aug_s = X_aug_s[perm_s], y_aug_s[perm_s]

    params_s = dict(best_params); params_s['random_state'] = seed
    xgb_defended_s = xgb.XGBClassifier(**params_s)
    xgb_defended_s.fit(X_aug_s, y_aug_s, verbose=False)

    clean_f1_s = f1_score(y_test, xgb_defended_s.predict(X_test_scaled), average='weighted')
    pred_pgd_s = xgb_defended_s.predict(X_adv_pgd_test)
    pgd_evasion_s = (pred_pgd_s[attack_mask_test] == 0).mean()

    ppo_bud_evasion_s, ppo_bud_perclass_s = subsample_ppo_evasion(
        ppo_model_budgeted, xgb_defended_s, 0.1, X_eval_sub, y_eval_sub)
    ppo_unc_evasion_s, ppo_unc_perclass_s = subsample_ppo_evasion(
        ppo_model_unconstrained, xgb_defended_s, None, X_eval_sub, y_eval_sub)

    seed_results.append({
        'seed': seed, 'clean_test_f1': clean_f1_s, 'pgd_evasion': pgd_evasion_s,
        'ppo_budgeted_evasion': ppo_bud_evasion_s, 'ppo_budgeted_per_class': ppo_bud_perclass_s,
        'ppo_unconstrained_evasion': ppo_unc_evasion_s, 'ppo_unconstrained_per_class': ppo_unc_perclass_s,
    })
    print(f"  Clean F1: {clean_f1_s:.4f} | PGD evasion: {pgd_evasion_s:.4f} | "
          f"PPO-budgeted evasion (subsample): {ppo_bud_evasion_s:.4f} | "
          f"PPO-unconstrained evasion (subsample): {ppo_unc_evasion_s:.4f}")
    print(f"  backdoor unconstrained evasion this seed: {ppo_unc_perclass_s['backdoor']:.4f}")


## Seed-robustness summary

In [ ]:
clean_f1s = [r['clean_test_f1'] for r in seed_results]
pgd_evs = [r['pgd_evasion'] for r in seed_results]
ppo_bud_evs = [r['ppo_budgeted_evasion'] for r in seed_results]
ppo_unc_evs = [r['ppo_unconstrained_evasion'] for r in seed_results]
backdoor_unc_evs = [r['ppo_unconstrained_per_class']['backdoor'] for r in seed_results]

print('=' * 70)
print(f"{'Metric':<38} {'Mean':>10} {'Std':>10}")
print('-' * 70)
print(f"{'Clean Test F1':<38} {np.mean(clean_f1s):>10.4f} {np.std(clean_f1s):>10.4f}")
print(f"{'PGD Evasion (full set)':<38} {np.mean(pgd_evs):>10.4f} {np.std(pgd_evs):>10.4f}")
print(f"{'PPO budget-matched Evasion (subs.)':<38} {np.mean(ppo_bud_evs):>10.4f} {np.std(ppo_bud_evs):>10.4f}")
print(f"{'PPO unconstrained Evasion (subs.)':<38} {np.mean(ppo_unc_evs):>10.4f} {np.std(ppo_unc_evs):>10.4f}")
print(f"{'backdoor unconstrained Evasion':<38} {np.mean(backdoor_unc_evs):>10.4f} {np.std(backdoor_unc_evs):>10.4f}")
print('=' * 70)

# CHANGED: compare against THIS run's actual single-run standard-model backdoor rate
# (computed in Section 9 above), not a hardcoded value from the tuned-model notebook --
# that hardcoded 0.8511 was specific to the Optuna-tuned model and would silently be
# the wrong comparison point here.
_backdoor_standard_rate = evasion_unc_std[1]['evaded'] / evasion_unc_std[1]['total']
n_backdoor_worse = sum(1 for e in backdoor_unc_evs if e > _backdoor_standard_rate)
print(f"\nbackdoor unconstrained evasion exceeded the STANDARD model's rate "
      f"({_backdoor_standard_rate:.4f}) in {n_backdoor_worse}/{len(SEED_LIST)} seeds.")
if n_backdoor_worse >= 4:
    print("-> The 'defence backfires for backdoor under unconstrained PPO' finding is "
          "SEED-ROBUST -- safe to report as a real result.")
elif n_backdoor_worse <= 1:
    print("-> The original single-seed result looks like noise -- do NOT report the "
          "'backfire' framing without qualification; the mean suggests roughly flat, "
          "not worse.")
else:
    print("-> Mixed across seeds -- report as 'inconsistent across seeds, mean evasion "
          "X +/- Y' rather than a clean backfire claim.")

joblib.dump(seed_results, f'defence1_seed_robustness_baseline_{PIPELINE_VERSION}.pkl')
print(f"\nSaved: defence1_seed_robustness_baseline_{PIPELINE_VERSION}.pkl")
